# ODS v2: Policy

Builds 22 ods tables in `ods_v2`: `Policy`, `PolicyVersion`, `PolicyTravel`, `PolicyTravelTransaction`, `PolicyMotorProposer`, `PolicyMotorVehicle`, `PolicyHome`, `PolicySale`, `PolicyRenewal`, `PolicyMonthEnd`, `PolicyCancellation`, `PolicyTransaction`, `PolicyMta`, `PolicyClaimMotor`, `PolicyClaimHome`, `ClientPortfolio`, `PolicyDocumentEvent`, `RenewalPortalEvent`, `DocumentWorkflowStatus`, `DocumentUpload`, `DocumentChaseHome`, `DocumentChaseMotorVan`.

Depends on no other build notebook.

Generated by `ods_build/v2/build_notebook.py` from `entities/*.yaml`: one SQL statement per cell.

**If a cell fails**

1. Read the failing cell's header: what it builds, what it reads, its grain, key and dedupe rule.
2. Fix the cause. Most failures are a source change (a renamed or missing column) or a broken key (duplicates).
   The checks cell at the end names the table and the likely cause.
3. Re-run from the failed cell: every cell rebuilds its table completely (`create or replace`), so re-running is
   always safe. After a session restart, run the first three cells again first (parameters, setup, start log).
4. **Copy the fix back into the config** (`ods_build/v2/entities/*.yaml`) and regenerate. This notebook is generated:
   a fix made only here is lost at the next generation (the drift check lists such cells).


In [ ]:
%%pyspark
# Parameters (set by ODS_v2_Run; leave blank when running by hand)
batch_id   = ""
source_set = ""


In [ ]:
%%pyspark
%run ODS_v2_00_Setup

In [ ]:
%%pyspark
ods_run_log("10_Policy", "started")


## Policy


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.Policy                                                                       */
/*  Description : Current state of every policy: the highest RecordVersion per policy code (RecordVersi */
/*              : onIsLatest is known to be                                                             */
/*              : unreliable, so it is carried, not used). ClientCode falls back to the first 6 charact */
/*              : ers of a 9-character code.                                                            */
/*              : EdwPolicyKey is the warehouse key, kept for reconciliation.                           */
/*              :                                                                                       */
/*              : Grain: one row per policy code, its latest version                                    */
/*              : Business key: PolicyCode   Surrogate key: PolicyId                                    */
/*              : History: current (all versions in PolicyVersion)   Cut-off: none                      */
/*              : Source: edw.tbl_ref_policy                                                            */
/*              : Reads: edw.tbl_ref_policy                                                             */
/******************************************************************************************************/
create or replace table ${ods.target}.Policy as
select  row_number() over (order by PolicyCode)                     as PolicyId,
        x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  nullif(upper(trim(p.PolicyCode)), '')                                              as PolicyCode,
                coalesce(nullif(upper(trim(p.ClientCode)), ''),
                         case when length(trim(p.PolicyCode)) = 9 then upper(left(trim(p.PolicyCode), 6)) end)
                                                                                as ClientCode,
                p.PolicyKey                                                     as EdwPolicyKey,
                p.ClientKey                                                     as EdwClientKey,
                p.PolicyTypeCode,
                p.PolicyTypeDesc,
                p.PolicyTypeGroup,
                p.DeptCode,
                p.DeptDesc,
                p.DeptGroup,
                p.InsCoCode,
                p.InsCoName,
                nullif(upper(trim(p.InsPolNumber)), '')                                            as InsPolNumber,
                p.PolicyHandlerCode,
                p.PolicyStatusCode,
                p.PolicyStatusDesc,
                p.PolicySalesDate,
                p.PolicyInceptDate,
                p.PolicyRenewDate,
                p.PolicyLapsedDate,
                p.PolicyCancelDate,
                p.CurrencyCode,
                p.CCYGrossPremium,
                p.CCYCommission,
                p.CCYFees,
                p.CommissionRate,
                p.PaymentType,
                p.FinanceFlag,
                p.RenewOnlineEnabled,
                p.SchemeID,
                p.SchemeName,
                p.SchemeCode,
                p.SchemeProfile,
                p.TelematicsFlag,
                p.EDIFlag,
                p.PromoCodeAmount,
                p.PromoCodeDate,
                p.PromoCodeTreatment,
                p.SegmentCluster,
                p.LYKeyCareFlag,
                p.RecordVersion                                                 as LatestRecordVersion,
                p.RecordVersionIsLatest,
                p.RecordDateTimeStart                                           as LatestRecordDateTimeStart
        from    (
                select  *,
                        row_number() over (partition by upper(trim(PolicyCode)) order by RecordVersion desc) as rn
                from    ${ods.edw}.tbl_ref_policy
                where   nullif(trim(PolicyCode), '') is not null
                ) p
        where   p.rn = 1
        ) x

## PolicyVersion


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.PolicyVersion                                                                */
/*  Description : Every version of every policy with its validity interval: status, dates, insurer, sch */
/*              : eme and premium as they                                                               */
/*              : stood. For "what did this policy look like at time t" and status-change history.      */
/*              :                                                                                       */
/*              : Grain: one row per policy code and record version                                     */
/*              : Business key: PolicyCode + RecordVersion   Surrogate key: PolicyVersionId             */
/*              : History: versioned (RecordDateTimeStart / RecordDateTimeEnd)   Cut-off: none          */
/*              : Source: edw.tbl_ref_policy                                                            */
/*              : Reads: edw.tbl_ref_policy, ods_v2.Policy                                              */
/******************************************************************************************************/
create or replace table ${ods.target}.PolicyVersion as
select  row_number() over (order by PolicyCode, RecordVersion)      as PolicyVersionId,
        x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  p.PolicyId,
                nullif(upper(trim(v.PolicyCode)), '')                                              as PolicyCode,
                v.RecordVersion,
                v.RecordVersionIsLatest,
                v.RecordDateTimeStart,
                v.RecordDateTimeEnd,
                nullif(upper(trim(v.ClientCode)), '')                                              as ClientCode,
                v.PolicyTypeCode,
                v.PolicyTypeGroup,
                v.InsCoCode,
                v.InsCoName,
                nullif(upper(trim(v.InsPolNumber)), '')                                            as InsPolNumber,
                v.PolicyStatusCode,
                v.PolicyStatusDesc,
                v.PolicySalesDate,
                v.PolicyInceptDate,
                v.PolicyRenewDate,
                v.PolicyLapsedDate,
                v.PolicyCancelDate,
                v.CCYGrossPremium,
                v.CCYCommission,
                v.CCYFees,
                v.PaymentType,
                v.FinanceFlag,
                v.SchemeName,
                v.SchemeCode,
                v.RenewOnlineEnabled
        from    ${ods.edw}.tbl_ref_policy   v
        join    ${ods.target}.Policy        p   on p.PolicyCode = upper(trim(v.PolicyCode))
        ) x

## PolicyTravel


In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_travel_policy_link                                                   */
/*  Description : One Relay travel policy per insurer policy number (the lowest PolicyId where several  */
/*              : share it).                                                                            */
/*              :                                                                                       */
/*              : Reads: ods_v2.Policy                                                                  */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_travel_policy_link as
select  InsPolNumber, min(PolicyId) as PolicyId
from    ${ods.target}.Policy
where   PolicyTypeGroup = 'Travel'
and     InsPolNumber is not null
group by InsPolNumber

In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.PolicyTravel                                                                 */
/*  Description : Travel policy detail from the Mapfre feed (latest UpdatedDate per policy), as noteboo */
/*              : k 01. Joined to the Relay                                                             */
/*              : policy by InsPolNumber = PolicyReference (MAPCHILL...). Name, address, email and phon */
/*              : e are not taken.                                                                      */
/*              :                                                                                       */
/*              : Grain: one row per Mapfre travel policy id, its latest update                         */
/*              : Business key: MapfrePolicyId                                                          */
/*              : History: current   Cut-off: none                                                      */
/*              : Source: dlk.ext_travel_policy_trans + dlk.ext_travel_policy                           */
/*              : Reads: dlk.ext_travel_policy_trans, dlk.ext_travel_policy, stg.odsv2_travel_policy_li */
/*              : nk                                                                                    */
/******************************************************************************************************/
create or replace table ${ods.target}.PolicyTravel as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  l.PolicyId,
                t.PolicyId                                                      as MapfrePolicyId,
                concat('M', t.MapfreCustomerId)                                 as ClientCode,
                t.MapfreCustomerId,
                t.QuoteId                                                       as MapfreQuoteId,
                nullif(upper(trim(t.PolicyReference)), '')                                         as InsPolicyCode,
                t.ProductName,
                t.PolicyTypeName,
                t.CreatedDate,
                t.UpdatedDate                                                   as LastUpdatedDate,
                t.PurchaseDate                                                  as SalesDate,
                t.StartDate                                                     as PolicyStartDate,
                t.EndDate                                                       as PolicyEndDate,
                t.TotalPremium,
                t.TravelPremium,
                t.MedicalPremium,
                t.ExpectedFee,
                t.AdminFee,
                t.HazActivityCount,
                t.MedicalCount,
                t.MKT_OPT_IN,
                b.Source                                                        as SalesChannel,
                t.VoidFlag,
                t.VoidDate,
                b.RenewalFlag,
                b.ExpectedCommission,
                b.Promocode
        from    (
                select  *, row_number() over (partition by PolicyId order by UpdatedDate desc) as rn
                from    ${ods.dlk}.ext_travel_policy_trans
                ) t
        left join ${ods.dlk}.ext_travel_policy          b   on b.PolicyId = t.PolicyId
        left join ${ods.stg}.odsv2_travel_policy_link   l   on l.InsPolNumber = upper(trim(t.PolicyReference))
        where   t.rn = 1
        ) x

## PolicyTravelTransaction


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.PolicyTravelTransaction                                                      */
/*  Description : Every Mapfre travel policy transaction (new business, renewals, voids, amendments) wi */
/*              : th its record type, not only                                                          */
/*              : the latest per policy (that is PolicyTravel). ClientCode is 'M' + MapfreCustomerId, t */
/*              : he form the SCV link                                                                  */
/*              : (ods.scveadmlinkfinal, system 5) holds. Name, address, email and phone are not taken. */
/*              :                                                                                       */
/*              : Grain: one row per Mapfre travel policy transaction                                   */
/*              : Business key: TravelTransactionKey                                                    */
/*              : History: event   Cut-off: UpdatedDate                                                 */
/*              : Source: dlk.ext_travel_policy_trans                                                   */
/*              : Reads: dlk.ext_travel_policy_trans, stg.odsv2_travel_policy_link                      */
/******************************************************************************************************/
create or replace table ${ods.target}.PolicyTravelTransaction as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  t.EXT_Travel_Policy_TransKey                                    as TravelTransactionKey,
                l.PolicyId,
                t.PolicyId                                                      as MapfrePolicyId,
                concat('M', t.MapfreCustomerId)                                 as ClientCode,
                t.MapfreCustomerId,
                t.QuoteId                                                       as MapfreQuoteId,
                t.PreviousQuoteId                                               as MapfrePreviousQuoteId,
                t.RecordType,
                t.Source,
                t.RenewalFlag,
                nullif(upper(trim(t.PolicyReference)), '')                                         as InsPolicyCode,
                t.ProductName,
                t.PolicyTypeName,
                t.CreatedDate,
                t.UpdatedDate,
                t.PurchaseDate,
                t.StartDate,
                t.EndDate,
                t.Promocode,
                t.TotalPremium,
                t.TravelPremium,
                t.MedicalPremium,
                t.ExpectedFee,
                t.AdminFee,
                t.HazActivityCount,
                t.MedicalCount,
                t.MKT_OPT_IN,
                t.VoidFlag,
                t.VoidDate
        from    ${ods.dlk}.ext_travel_policy_trans              t
        left join ${ods.stg}.odsv2_travel_policy_link   l   on l.InsPolNumber = upper(trim(t.PolicyReference))
        where   t.UpdatedDate >= date '${ods.cutoff_date}'
        ) x

## PolicyMotorProposer


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.PolicyMotorProposer                                                          */
/*  Description : Motor proposer detail per policy version, carried from the current ods (loaded from t */
/*              : he SQL Server warehouse;                                                              */
/*              : there is no tbl_ref_policy_motor in Fabric). To be rebuilt from dlk.rbsdata2_motor_*  */
/*              : with the DWH logic.                                                                   */
/*              :                                                                                       */
/*              : Grain: one row per policy code and record version                                     */
/*              : Business key: PolicyCode + RecordVersion                                              */
/*              : History: versioned (RecordVersion)   Cut-off: none                                    */
/*              : Source: ods.policymotorproposer (carried forward; no edw source in Fabric)            */
/*              : Reads: ods.policymotorproposer, ods_v2.Policy                                         */
/******************************************************************************************************/
create or replace table ${ods.target}.PolicyMotorProposer as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  p.PolicyId,
                nullif(upper(trim(m.PolicyCode)), '')                                              as PolicyCode,
                m.RecordVersion,
                m.ProposerResidentSinceDate,
                m.ProposerLicenceTypeDesc,
                m.ProposerLicenceIssueCountry,
                m.ProposerLicenceIssueDate,
                m.ProposerEmploymentStatusDesc,
                m.ProposerOccupationDesc,
                m.ProposerNCDYears,
                m.ProposerNCDProtection,
                m.ProposerNCDCountry,
                m.ProposerPreviousInsFlag,
                m.ProposerPreviouslyImposedTermsFlag,
                m.ProposerPreviouslyRefusedFlag,
                m.ProposerNonMotoringConvictionsFlag,
                m.ndVehicleFlag                                                 as SecondVehicleFlag,
                m.RiskProfileBand,
                m.ProposerAge,
                m.ProposerNDEYears
        from    (select distinct * from ${ods.ods_carry}.policymotorproposer) m
        left join ${ods.target}.Policy  p   on p.PolicyCode = upper(trim(m.PolicyCode))
        ) x

## PolicyMotorVehicle


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.PolicyMotorVehicle                                                           */
/*  Description : Motor vehicle detail per policy version, carried from the current ods. The registrati */
/*              : on is held only as a                                                                  */
/*              : salted hash (VehicleRegKey).                                                          */
/*              :                                                                                       */
/*              : Grain: one row per policy code and record version                                     */
/*              : Business key: PolicyCode + RecordVersion                                              */
/*              : History: versioned (RecordVersion)   Cut-off: none                                    */
/*              : Source: ods.policymotorvehicle (carried forward; no edw source in Fabric)             */
/*              : Reads: ods.policymotorvehicle, ods_v2.Policy                                          */
/******************************************************************************************************/
create or replace table ${ods.target}.PolicyMotorVehicle as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  p.PolicyId,
                nullif(upper(trim(v.PolicyCode)), '')                                              as PolicyCode,
                v.RecordVersion,
                v.VehicleManufacturer,
                v.VehicleModel,
                v.VehicleManufacturedYearFrom,
                v.VehicleManufacturedYearTo,
                v.VehicleType,
                v.VehicleFuelType,
                v.VehicleEngineCC,
                v.VehicleImportedFlag,
                v.VehicleAlarmFlag,
                v.VehicleTrailerFlag,
                v.VehicleExcessAmount,
                v.VehiclePurchaseDate,
                v.VehicleValue,
                v.VehicleMileageBusiness,
                v.VehicleMileagePrivate,
                v.VehicleRegIrelandFlag,
                ods_hash_id(v.VehicleRegNumber, '')                                     as VehicleRegKey,
                v.VehicleRegDate,
                v.VehicleCoverType,
                v.VehicleValidDrivers,
                v.VehicleNoOfLiveDrivers,
                v.VehicleClassOfUse,
                v.VehicleMainUseAreaCode,
                v.VehicleMainUseArea,
                v.VehicleMainUseAreaCounty,
                v.VehicleKeptOvernightAreaCode,
                v.VehicleKeptOvernightArea,
                v.VehicleKeptOvernightCounty,
                v.VehicleLHDFlag
        from    (select distinct * from ${ods.ods_carry}.policymotorvehicle) v
        left join ${ods.target}.Policy  p   on p.PolicyCode = upper(trim(v.PolicyCode))
        ) x

## PolicyHome


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.PolicyHome                                                                   */
/*  Description : Home risk detail per policy version, carried from the current ods. The Eircode is red */
/*              : uced to its routing key                                                               */
/*              : (HomeEircodeArea); the full code is not held.                                         */
/*              :                                                                                       */
/*              : Grain: one row per policy code and record version                                     */
/*              : Business key: PolicyCode + RecordVersion                                              */
/*              : History: versioned (RecordVersion)   Cut-off: none                                    */
/*              : Source: ods.policyhome (carried forward; no edw source in Fabric)                     */
/*              : Reads: ods.policyhome, ods_v2.Policy                                                  */
/******************************************************************************************************/
create or replace table ${ods.target}.PolicyHome as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  p.PolicyId,
                nullif(upper(trim(h.PolicyCode)), '')                                              as PolicyCode,
                h.RecordVersion,
                h.HomeFirstTimeBuyerFlag,
                h.HomeNCDYears,
                h.HomeNCDYearsGranted,
                h.HomeCCRequiredSumInsured,
                h.HomeCCGrantedSumInsured,
                h.HomeCCAccidentalDamageGrantedFlag,
                h.HomeBCRequiredSumInsured,
                h.HomeBCGrantedSumInsured,
                h.HomeBCAccidentalDamageGrantedFlag,
                h.HomeSICNoOfItemsCovered,
                h.HomeSICRequiredSumInsured,
                h.HomeSICGrantedSumInsured,
                h.HomePropertyType,
                h.HomeConstructionYear,
                h.HomeNoOfBathrooms,
                h.HomeNoOfBedrooms,
                h.HomeNoOfSmokeAlarms,
                h.HomeHeatingType,
                h.HomeRoofConstructionType,
                h.HomeRoofNonStandardPercentage,
                h.HomeLockType,
                h.HomeAlarmType,
                h.HomeNeighbourhoodWatchFlag,
                h.HomeProposerType,
                h.HomeResidenceType,
                h.HomeOccupiedDuringDayFlag,
                ods_eircode_area(h.HomeAddressPostCode)                             as HomeEircodeArea
        from    (select distinct * from ${ods.ods_carry}.policyhome) h
        left join ${ods.target}.Policy  p   on p.PolicyCode = upper(trim(h.PolicyCode))
        ) x

## PolicySale


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.PolicySale                                                                   */
/*  Description : Sale postings (new business, renewals, rebooks) for all products. Lead / handler pers */
/*              : on names and the email                                                                */
/*              : address are not taken; the registration is a salted hash.                             */
/*              :                                                                                       */
/*              : Grain: one row per policy code and reporting sale date                                */
/*              : Business key: PolicyCode + ReportingSaleDate                                          */
/*              : History: event (all sales, no cut-off)   Cut-off: none                                */
/*              : Source: edw.tbl_fact_policy_sales                                                     */
/*              : Reads: edw.tbl_fact_policy_sales, ods_v2.Policy                                       */
/******************************************************************************************************/
create or replace table ${ods.target}.PolicySale as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  p.PolicyId,
                nullif(upper(trim(s.PolicyCode)), '')                                              as PolicyCode,
                s.ReportingSaleDate,
                s.EffectiveDate,
                s.WorkDate,
                s.WorkWeek,
                s.ReportingSaleType,
                s.ReportingSaleCategory,
                s.PolicyTypeCode,
                s.PolicyTypeDesc,
                s.PolicyTypeGroup,
                s.PolicyStatusCode,
                s.PolicyStatusDesc,
                s.PolicyCloseNum,
                s.PolicyRenewDateAdj,
                s.PolicyRenewMonth,
                s.PolicyKeyEffDate                                              as EdwPolicyKeyEffDate,
                s.PolicyInceptDate,
                nullif(upper(trim(s.LYPolicyCode)), '')                                            as LYPolicyCode,
                s.LYReportingSaleDate,
                nullif(upper(trim(s.OLDPolicyCode)), '')                                           as OldPolicyCode,
                s.RiskProfileBand,
                s.InsCoCode,
                s.InsCoName,
                ods_redact(nullif(upper(trim(s.InsPolNumber)), ''))                                    as InsPolNumber,
                s.FinanceFlag,
                s.RenewalTransferFlag,
                s.MTARebookedFlag,
                nullif(lower(trim(s.QuoteQueryGUID)), '')                                          as QuoteQueryGuid,
                s.Channel,
                s.SubChannel,
                s.ChannelCall,
                s.ChannelWeb,
                s.ChannelXSell,
                s.ChannelOnLine,
                s.OrgID,
                s.SaleCount,
                s.EURGrossPremium,
                s.EURNetPremium,
                s.EURCommission,
                s.EURFees,
                s.SchemeName,
                s.SchemeCode,
                s.SchemeProfile,
                s.DeptGroup,
                s.CloserEventDate,
                s.DiaryPaymentTypeTY,
                s.DiaryPaymentTypeLY,
                s.RenewalsPortal,
                s.RenewalOnlineEnabled,
                s.VehicleCoverType,
                ods_hash_id(s.VehicleRegNumber, '')                                     as VehicleRegKey,
                s.CampaignKey,
                s.CampaignName,
                s.MFQQuoteSummaryID,
                s.MFQCreatedDatetime,
                s.MFQMatchType,
                s.MFQQuoteType,
                s.HEQReference,
                s.HEQCreatedDateTime,
                s.HEQMatchType,
                s.HEQQuoteType,
                s.kci_flag                                                      as KeyCareFlag,
                s.kci_type                                                      as KeyCareType,
                s.kci_premium                                                   as KeyCarePremium,
                s.kci_commission                                                as KeyCareCommission,
                s.kci_workDate                                                  as KeyCareWorkDate
        from    ${ods.edw}.tbl_fact_policy_sales    s
        left join ${ods.target}.Policy              p   on p.PolicyCode = upper(trim(s.PolicyCode))
        ) x

## PolicyRenewal


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.PolicyRenewal                                                                */
/*  Description : One row per renewal cycle of a policy: last-year term, the offer / invite, the outcom */
/*              : e (this-year policy code),                                                            */
/*              : portal logins and payments. PolicyCode is the last-year (expiring) policy; TYPolicyCo */
/*              : de the renewed one.                                                                   */
/*              : Phone numbers are not taken; registrations are salted hashes.                         */
/*              :                                                                                       */
/*              : Grain: one row per policy code and last-year renew date                               */
/*              : Business key: PolicyCode + LYPolicyRenewDateAdj                                       */
/*              : History: event (all renewal cycles, no cut-off)   Cut-off: none                       */
/*              : Source: edw.tbl_fact_policy_renewals_rebuilt (identical copy, until tbl_fact_policy_r */
/*              : enewals is reinstated)                                                                */
/*              : Reads: edw.tbl_fact_policy_renewals_rebuilt, ods_v2.Policy                            */
/******************************************************************************************************/
create or replace table ${ods.target}.PolicyRenewal as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  p.PolicyId,
                nullif(upper(trim(r.PolicyCode)), '')                                              as PolicyCode,
                r.LYPolicyRenewDateAdj,
                r.RenewalMonth,
                r.LYEffectiveDate,
                r.LYPolicyTypeCode,
                r.LYPolicyTypeGroup,
                r.LYPolicyStatusDesc,
                r.LYReportingSaleType,
                r.LYReportingSaleCategory,
                r.LYReportingSaleDate,
                r.LYInsCoCode,
                r.LYInsCoName,
                r.LYSchemeName,
                r.LYEURGrossPremium,
                r.LYEURCommission,
                r.LYEURFees,
                ods_hash_id(r.LYVehicleRegNumber, '')                                   as LYVehicleRegKey,
                r.MTAFlag,
                r.RenEURPremInvite,
                r.RenEURPremInviteOrig,
                r.RenEURPremAlternative,
                r.RenEURPremOffer,
                r.RenEURFee,
                r.PolicyOfferNum,
                r.PolicyRetNum,
                r.PolicyRetType,
                r.OfferType,
                nullif(upper(trim(r.NewPolicyCode)), '')                                           as NewPolicyCode,
                r.NewPolicyCodeRule,
                nullif(upper(trim(r.TYPolicyCode)), '')                                            as TYPolicyCode,
                r.TYEffectiveDate,
                r.TYInsCoCode,
                r.TYInsCoName,
                r.TYSchemeName,
                r.TYReportingSaleType,
                r.TYReportingSaleCategory,
                r.TYReportingSaleDate,
                r.TYPolicyRenewDateAdj,
                r.TYPolicyStatusDesc,
                r.TYEURGrossPremium,
                r.TYEURCommission,
                r.TYEURFees,
                ods_hash_id(r.TYVehicleRegNumber, '')                                   as TYVehicleRegKey,
                r.PolicyCodeRevisedAtOffer,
                r.RiskProfileBand,
                r.ClientAgeBand,
                r.VehicleCoverType,
                r.Channel,
                r.SubChannel,
                r.RenewalsPortal,
                r.DiaryPaymentTypeTY,
                r.PromoCodeApplied,
                r.PromoCodeAmount,
                r.PromoCodeDate,
                r.PromoCodeTreatment,
                r.SuccessfulLoginCount,
                r.FailedLoginCount,
                r.SuccessfulPaymentCount,
                r.FailedPaymentCount,
                r.LastCallDateTime,
                r.MFQSale,
                r.MFQSaleDate,
                r.ClientTenure,
                r.ProductTenure,
                r.CreateDate                                                    as EdwCreateDate
        from    ${ods.edw}.tbl_fact_policy_renewals_rebuilt r
        left join ${ods.target}.Policy              p   on p.PolicyCode = upper(trim(r.PolicyCode))
        ) x

## PolicyMonthEnd


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.PolicyMonthEnd                                                               */
/*  Description : Month-end position of each policy: open / close counts, movements in the month, close */
/*              :  premium, status and the                                                              */
/*              : holder's banded attributes as at that month-end. Date of birth and postcode are not t */
/*              : aken.                                                                                 */
/*              :                                                                                       */
/*              : Grain: one row per policy code and month-end date                                     */
/*              : Business key: PolicyCode + EffectiveDate                                              */
/*              : History: snapshot (month-end)   Cut-off: EffectiveDate                                */
/*              : Source: edw.tbl_fact_policy_mvt                                                       */
/*              : Reads: edw.tbl_fact_policy_mvt, ods_v2.Policy                                         */
/******************************************************************************************************/
create or replace table ${ods.target}.PolicyMonthEnd as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  p.PolicyId,
                nullif(upper(trim(m.PolicyCode)), '')                                              as PolicyCode,
                m.EffectiveDate,
                nullif(upper(trim(m.ClientCode)), '')                                              as ClientCode,
                m.OpenNum,
                m.CloseNum,
                m.NewBusGrossNum,
                m.RenewGrossNum,
                m.MTAGrossNum,
                m.MTCGrossNum,
                m.DueRenewalNum,
                m.OpenCloseDesc,
                m.EURGrossPremClose,
                m.EURCommissionClose,
                m.EURFeesClose,
                m.PolicyStatusCode,
                m.PolicyStatusDesc,
                m.PolicyTypeGroup,
                m.ReportingSaleType,
                m.ReportingSaleCategory,
                m.ReportingSaleDate,
                m.PolicyInceptDate,
                m.PolicyRenewDate,
                m.PolicyRenewDateAdj,
                m.PolicyLapsedDate,
                m.PolicyLapsedFlag,
                m.PolicyCancelDate,
                m.PolicyCancelFlag,
                m.InsCoCode,
                m.InsCoName,
                m.FinanceFlag,
                m.ClientAge,
                m.ClientAgeBand,
                m.ClientGender,
                m.ClientCounty,
                m.ClientProvince,
                ods_hash_id(m.VehicleRegNumber, '')                                     as VehicleRegKey,
                m.VehicleManufacturer,
                m.VehicleModel,
                m.VehicleCoverType,
                m.VehicleFuelType,
                m.RiskProfileBand,
                m.ClientTenure,
                m.ProductTenure,
                m.InsurerTenure,
                m.ChannelGroup,
                m.DiaryPaymentTypeTY
        from    ${ods.edw}.tbl_fact_policy_mvt  m
        left join ${ods.target}.Policy          p   on p.PolicyCode = upper(trim(m.PolicyCode))
        where   m.EffectiveDate >= date '${ods.cutoff_date}'
        ) x

## PolicyCancellation


In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_policy_cancellation                                                  */
/*  Description : Cancellation rows with a generated id.                                                */
/*              :                                                                                       */
/*              : Reads: edw.tbl_fact_policy_mtc                                                        */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_policy_cancellation as
select  row_number() over (order by c.PolicyCode, c.EffectiveDate, c.ExtractDatetime, c.rn, c.TrCode, c.DrCode,
                                    c.ShortDescription)                         as PolicyCancellationId,
        c.*
from    ${ods.edw}.tbl_fact_policy_mtc c

In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.PolicyCancellation                                                           */
/*  Description : Mid-term cancellations with reason, diary flags and the money reversed. The source ha */
/*              : s no small natural key, so                                                            */
/*              : each row gets a generated id (ordered by policy, effective date, extract time, row).  */
/*              :                                                                                       */
/*              : Grain: one row per cancellation record (no natural key; generated key)                */
/*              : Business key: PolicyCancellationId                                                    */
/*              : History: event (no cut-off; small)   Cut-off: none                                    */
/*              : Source: edw.tbl_fact_policy_mtc                                                       */
/*              : Reads: stg.odsv2_policy_cancellation, ods_v2.Policy                                   */
/******************************************************************************************************/
create or replace table ${ods.target}.PolicyCancellation as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  c.PolicyCancellationId,
                p.PolicyId,
                nullif(upper(trim(c.PolicyCode)), '')                                              as PolicyCode,
                c.EffectiveDate,
                nullif(upper(trim(c.ClientCode)), '')                                              as ClientCode,
                c.PolicyTypeGroup,
                c.Channel,
                c.InsCoCode,
                c.InsCoName,
                c.ReportingSaleDate,
                c.ReportingSaleCategory,
                c.PolicyStatusCode,
                c.PolicyStatusDesc,
                c.PolicyInceptionDate,
                c.DaysOnCover,
                c.PolicyRenewDate,
                c.PolicyCancelDate,
                c.PolicyCancelEffDate,
                c.PolicyMTCDate,
                c.Rebooked,
                c.CXLPolicy,
                nullif(upper(trim(c.OldPolicyCode)), '')                                           as OldPolicyCode,
                c.OldPolicyStatusDesc,
                c.DiaryPaymentTypeTY,
                c.OriginalGWP,
                c.OriginalFee,
                c.CancelledGWP,
                c.OriginalIPP,
                c.CancelledIPP,
                c.DrCode,
                c.TrCode,
                c.PlCancelReason,
                c.Category,
                c.ShortDescription,
                c.FullDescription,
                c.CancellationDiaryFlag,
                c.CPFCAncellationArrearsFlag                                    as CPFCancellationArrearsFlag,
                c.EurBalance,
                c.MTCNetNum,
                c.EURCommissionMTC,
                c.EURFeesMTC,
                c.ExtractDatetime,
                c.WorkDate
        from    ${ods.stg}.odsv2_policy_cancellation c
        left join ${ods.target}.Policy              p   on p.PolicyCode = upper(trim(c.PolicyCode))
        ) x

## PolicyTransaction


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.PolicyTransaction                                                            */
/*  Description : Relay accounts archive (premium, fee, commission postings) with the premium type from */
/*              :  the warehouse mapping                                                                */
/*              : (joined with spaces in the Relay codes read as '-', as the warehouse and notebook 04  */
/*              : do).                                                                                  */
/*              : Relay yyyymmdd decimals are turned into dates. Exact duplicate rows in the source (ab */
/*              : out 80k) are dropped.                                                                 */
/*              :                                                                                       */
/*              : Grain: one row per Relay accounts transaction (exact duplicate rows removed)          */
/*              : Business key: PolicyCode + PostingDate + TransSeq + ArCode + ArNumber + ArReference + */
/*              :  ClientTransType + ClientAmount   Surrogate key: PolicyTransactionId                  */
/*              : History: event   Cut-off: PostingDate                                                 */
/*              : Source: dlk.rbsdata2_newarchive + edw.tbl_ref_mapping_bar_premium_type                */
/*              : Reads: dlk.rbsdata2_newarchive, edw.tbl_ref_mapping_bar_premium_type                  */
/******************************************************************************************************/
create or replace table ${ods.target}.PolicyTransaction as
select  row_number() over (order by PolicyCode, PostingDate, TransSeq, ArCode, ArNumber, ArReference, ClientTransType, ClientAmount)as PolicyTransactionId,
        x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  distinct
                nullif(upper(trim(a.ar_policy_code)), '')                                          as PolicyCode,
                nullif(upper(trim(a.ar_client_code)), '')                                          as ClientCode,
                case when a.ar_posting_date > 19000000 then to_date(cast(cast(a.ar_posting_date as bigint) as string), 'yyyyMMdd') end                                   as PostingDate,
                case when a.ar_effective_date > 19000000 then to_date(cast(cast(a.ar_effective_date as bigint) as string), 'yyyyMMdd') end                                 as EffectiveDate,
                a.ar_trans_seq                                                  as TransSeq,
                a.ar_code                                                       as ArCode,
                a.ar_number                                                     as ArNumber,
                ods_redact(a.ar_reference)                                          as ArReference,
                a.ar_cli_type                                                   as ClientTransType,
                a.ar_cli_amount                                                 as ClientAmount,
                a.ar_cli_net_premium                                            as ClientNetPremium,
                a.ar_cli_commission                                             as ClientCommission,
                a.ar_cli_fees                                                   as ClientFees,
                a.ar_cli_discount                                               as ClientDiscount,
                a.ar_annual_prem                                                as AnnualPremium,
                a.ar_com_type                                                   as InsurerTransType,
                a.ar_com_sta                                                    as InsurerStatus,
                a.ar_com_initial_net_prem                                       as InsurerInitialNetPremium,
                a.ar_com_initial_comm                                           as InsurerInitialCommission,
                a.ar_confirmed_prov                                             as ConfirmedProvisional,
                a.ar_return_sub_type                                            as ReturnSubType,
                a.ar_insco                                                      as InsCoCode,
                a.ar_pol_type                                                   as PolicyType,
                a.ar_new_flag                                                   as NewFlag,
                a.ar_show_on_report                                             as ShowOnReport,
                a.ar_dtype                                                      as DType,
                a.ar_trans_description                                          as TransDescription,
                a.ar_addon_ref                                                  as AddonRef,
                a.ar_department                                                 as Department,
                a.ar_handler                                                    as HandlerCode,
                a.ar_user                                                       as UserCode,
                a.ar_paymenttype                                                as PaymentType,
                a.ar_paymentmethod                                              as PaymentMethod,
                a.ar_lapse_reason_type                                          as LapseReasonType,
                a.ar_lapse_reason_code                                          as LapseReasonCode,
                m.PremiumType,
                m.PremiumTypeGroup
        from    ${ods.dlk}.rbsdata2_newarchive                  a
        left join ${ods.edw}.tbl_ref_mapping_bar_premium_type   m          -- Relay codes hold spaces where the mapping has '-'
                on  m.ar_cli_type        = replace(a.ar_cli_type, ' ', '-')
                and m.ar_com_type        = replace(a.ar_com_type, ' ', '-')
                and m.ar_return_sub_type = replace(a.ar_return_sub_type, ' ', '-')
                and m.ar_com_sta         = replace(a.ar_com_sta, ' ', '-')
                and m.ar_confirmed_prov  = replace(a.ar_confirmed_prov, ' ', '-')
        where   case when a.ar_posting_date > 19000000 then to_date(cast(cast(a.ar_posting_date as bigint) as string), 'yyyyMMdd') end >= date '${ods.cutoff_date}'
        ) x

## PolicyMta


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.PolicyMta                                                                    */
/*  Description : Relay mid-term adjustment history: what changed (the Change_* / Amend_* flags), premi */
/*              : um and fee, effective                                                                 */
/*              : window. Free-text notes and snapshots are not taken (they can hold personal detail).  */
/*              :                                                                                       */
/*              : Grain: one row per policy, history sequence and history type                          */
/*              : Business key: PolicyCode + HistorySeq + HistoryType                                   */
/*              : History: event   Cut-off: ProcessedDate                                               */
/*              : Source: dlk.rbsdata2_mta_history                                                      */
/*              : Reads: dlk.rbsdata2_mta_history                                                       */
/******************************************************************************************************/
create or replace table ${ods.target}.PolicyMta as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  nullif(upper(trim(h.Hi_pol_code)), '')                                             as PolicyCode,
                h.Hi_seq                                                        as HistorySeq,
                h.Hi_type                                                       as HistoryType,
                h.hi_sub_type                                                   as HistorySubType,
                h.Hi_trcode                                                     as TrCode,
                case when h.Hi_date > 19000000 then to_date(cast(cast(h.Hi_date as bigint) as string), 'yyyyMMdd') end                                           as HistoryDate,
                h.Processed_Date                                                as ProcessedDate,
                h.Effective_Start_Date                                          as EffectiveStartDate,
                h.Effective_End_Date                                            as EffectiveEndDate,
                h.MTA_Type                                                      as MtaType,
                h.Sub_Type                                                      as MtaSubType,
                h.Premium,
                h.Fee,
                h.Hi_amount                                                     as Amount,
                h.Hi_an_prem                                                    as AnnualPremium,
                h.Hi_Manual                                                     as ManualFlag,
                h.Change_of_Policyholder_Details                                as ChangeOfPolicyholderDetails,
                h.Change_Address                                                as ChangeAddress,
                h.Amend_Licence                                                 as AmendLicence,
                h.Amend_Convictions_Proposer                                    as AmendConvictionsProposer,
                h.Change_NCD_Protection                                         as ChangeNcdProtection,
                h.Change_Policy_Excess                                          as ChangePolicyExcess,
                h.Permanent_Substitution                                        as PermanentSubstitution,
                h.Change_of_Cover                                               as ChangeOfCover,
                h.Change_Class_of_Use                                           as ChangeClassOfUse,
                h.Change_Registration_No                                        as ChangeRegistrationFlag,
                h.Change_Driving_Option                                         as ChangeDrivingOption,
                h.Change_Area_of_Use                                            as ChangeAreaOfUse,
                h.Change_NCD                                                    as ChangeNcd,
                h.Amend_Additional_Items                                        as AmendAdditionalItems,
                h.Amend_Vehicle_Details                                         as AmendVehicleDetails,
                h.Permanent_Additional_Drivers                                  as PermanentAdditionalDrivers,
                h.Amend_Licence_Details                                         as AmendLicenceDetails,
                h.Amend_Convictions_Other                                       as AmendConvictionsOther,
                h.Change_of_Additional_Driver_Details                           as ChangeOfAdditionalDriverDetails,
                h.Amend_2nd_Car_Discount                                        as Amend2ndCarDiscount
        from    ${ods.dlk}.rbsdata2_mta_history h
        where   h.Processed_Date >= date '${ods.cutoff_date}'
        ) x

## PolicyClaimMotor


In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_motor_claim_slots                                                    */
/*  Description : The five claim slots as rows, with the extract time used to keep the latest.          */
/*              :                                                                                       */
/*              : Reads: dlk.rbsdata2_motor_clm                                                         */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_motor_claim_slots as
select va_clm_plcode2 as PolicyCode, 1 as ClaimSlot, va_clm_date_1 as ClaimDate, va_clm_type_1 as ClaimType, va_clm_value_1 as ClaimValue, va_clm_fault_1 as OwnFaultFlag, va_clm_inj_1 as InjuryFlag, va_clm_ncb_affected_1 as NcbAffected, va_clm_settled_1 as Settled, coalesce(`$ExtractDateTime`, ReplicationDateTime) as ExtractDateTime from ${ods.dlk}.rbsdata2_motor_clm where va_clm_date_1 > 0
union all
select va_clm_plcode2, 2, va_clm_date_2, va_clm_type_2, va_clm_value_2, va_clm_fault_2, va_clm_inj_2, va_clm_ncb_affected_2, va_clm_settled_2, coalesce(`$ExtractDateTime`, ReplicationDateTime) from ${ods.dlk}.rbsdata2_motor_clm where va_clm_date_2 > 0
union all
select va_clm_plcode2, 3, va_clm_date_3, va_clm_type_3, va_clm_value_3, va_clm_fault_3, va_clm_inj_3, va_clm_ncb_affected_3, va_clm_settled_3, coalesce(`$ExtractDateTime`, ReplicationDateTime) from ${ods.dlk}.rbsdata2_motor_clm where va_clm_date_3 > 0
union all
select va_clm_plcode2, 4, va_clm_date_4, va_clm_type_4, va_clm_value_4, va_clm_fault_4, va_clm_inj_4, va_clm_ncb_affected_4, va_clm_settled_4, coalesce(`$ExtractDateTime`, ReplicationDateTime) from ${ods.dlk}.rbsdata2_motor_clm where va_clm_date_4 > 0
union all
select va_clm_plcode2, 5, va_clm_date_5, va_clm_type_5, va_clm_value_5, va_clm_fault_5, va_clm_inj_5, va_clm_ncb_affected_5, va_clm_settled_5, coalesce(`$ExtractDateTime`, ReplicationDateTime) from ${ods.dlk}.rbsdata2_motor_clm where va_clm_date_5 > 0

In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.PolicyClaimMotor                                                             */
/*  Description : Motor claims declared on Relay policies: the five claim slots per record turned into  */
/*              : rows (as notebook 04),                                                                */
/*              : latest extract per policy and slot. ClaimDate is null where Relay holds an impossible */
/*              :  date (e.g. 30 Feb);                                                                  */
/*              : ClaimDateRaw keeps the yyyymmdd value as held and is the key.                         */
/*              :                                                                                       */
/*              : Grain: one row per policy, claim date, claim type and value                           */
/*              : Business key: PolicyCode + ClaimDateRaw + ClaimType + ClaimValue + ClaimSlot          */
/*              : History: event (no cut-off; claims history is a risk attribute)   Cut-off: none       */
/*              : Source: dlk.rbsdata2_motor_clm                                                        */
/*              : Reads: stg.odsv2_motor_claim_slots, ods_v2.Policy                                     */
/******************************************************************************************************/
create or replace table ${ods.target}.PolicyClaimMotor as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  p.PolicyId,
                nullif(upper(trim(c.PolicyCode)), '')                                              as PolicyCode,
                c.ClaimSlot,
                case when c.ClaimDate > 19000000 then to_date(cast(cast(c.ClaimDate as bigint) as string), 'yyyyMMdd') end                                         as ClaimDate,
                c.ClaimDate                                                     as ClaimDateRaw,
                c.ClaimType,
                c.ClaimValue,
                c.OwnFaultFlag,
                c.InjuryFlag,
                c.NcbAffected,
                c.Settled,
                c.ExtractDateTime
        from    (
                select  *,
                        row_number() over (partition by upper(trim(PolicyCode)), ClaimDate, ClaimType, ClaimValue, ClaimSlot
                                           order by ExtractDateTime desc) as rn
                from    ${ods.stg}.odsv2_motor_claim_slots
                ) c
        left join ${ods.target}.Policy  p   on p.PolicyCode = upper(trim(c.PolicyCode))
        where   c.rn = 1
        ) x

## PolicyClaimHome


In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_home_risk_policy                                                     */
/*  Description : One policy code per home risk id (the latest transaction).                            */
/*              :                                                                                       */
/*              : Reads: dlk.ext_home_qs_policydetails                                                  */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_home_risk_policy as
select  upper(RiskId) as HomeRiskId, PolicyCode
from    (
        select  RiskId, PolicyCode,
                row_number() over (partition by upper(RiskId) order by TransactionDate desc) as rn
        from    ${ods.dlk}.ext_home_qs_policydetails
        where   nullif(trim(PolicyCode), '') is not null
        ) x
where   rn = 1

In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.PolicyClaimHome                                                              */
/*  Description : Home claims declared in the home quotation system, linked to the policy code through  */
/*              : the risk id.                                                                          */
/*              :                                                                                       */
/*              : Grain: one row per home claim id                                                      */
/*              : Business key: HomeClaimId                                                             */
/*              : History: event (no cut-off)   Cut-off: none                                           */
/*              : Source: dlk.ext_home_qs_claimdetails + dlk.ext_home_qs_policydetails                  */
/*              : Reads: dlk.ext_home_qs_claimdetails, stg.odsv2_home_risk_policy, ods_v2.Policy        */
/******************************************************************************************************/
create or replace table ${ods.target}.PolicyClaimHome as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  c.HomeClaimId,
                p.PolicyId,
                nullif(upper(trim(r.PolicyCode)), '')                                              as PolicyCode,
                upper(c.HomeRiskId)                                             as HomeRiskId,
                c.HomeClaimType,
                c.OccurredAt,
                c.HomeClaimAmount,
                c.Settled,
                c.HomeClaimCircumstance
        from    (
                select  *, row_number() over (partition by HomeClaimId order by `$ExtractDateTime` desc) as rn
                from    ${ods.dlk}.ext_home_qs_claimdetails
                ) c
        left join ${ods.stg}.odsv2_home_risk_policy r   on r.HomeRiskId = upper(c.HomeRiskId)
        left join ${ods.target}.Policy              p   on p.PolicyCode = upper(trim(r.PolicyCode))
        where   c.rn = 1
        ) x

## ClientPortfolio


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.ClientPortfolio                                                              */
/*  Description : Relay portfolio (client) to policy code, as notebook 05 (PolicyCode = portfolio code  */
/*              : + 3-digit sequence).                                                                  */
/*              :                                                                                       */
/*              : Grain: one row per Relay portfolio and policy code                                    */
/*              : Business key: PortfolioId + PolicyCode                                                */
/*              : History: current   Cut-off: none                                                      */
/*              : Source: dlk.relay_policy + dlk.relay_portfolio                                        */
/*              : Reads: dlk.relay_policy, dlk.relay_portfolio, ods_v2.Policy                           */
/******************************************************************************************************/
create or replace table ${ods.target}.ClientPortfolio as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  e.PortfolioId,
                nullif(upper(trim(e.Code)), '')                                                    as ClientCode,
                concat(upper(trim(e.Code)), lpad(cast(d.PolicySequence as string), 3, '0'))
                                                                                as PolicyCode,
                max(p.PolicyId)                                                 as PolicyId
        from    ${ods.dlk}.relay_policy     d
        join    ${ods.dlk}.relay_portfolio  e   on e.PortfolioId = d.PortfolioId
        left join ${ods.target}.Policy      p   on p.PolicyCode = concat(upper(trim(e.Code)), lpad(cast(d.PolicySequence as string), 3, '0'))
        group by e.PortfolioId, upper(trim(e.Code)), concat(upper(trim(e.Code)), lpad(cast(d.PolicySequence as string), 3, '0'))
        ) x

## PolicyDocumentEvent


In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_relay_document_event                                                 */
/*  Description : Relay events with their document, business process and event type name.               */
/*              :                                                                                       */
/*              : Reads: dlk.relay_eventmodel_event, dlk.relay_eventmodel_documentevent, dlk.relay_docu */
/*              : mentation_document, dlk.relay_reporting_relaycodelist                                 */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_relay_document_event as
select  distinct
        a.EventId,
        c.DocumentId,
        b.GeneratedByBusinessProcess                                as BusinessProcessCode,
        case b.GeneratedByBusinessProcess
             when 0 then 'Unknown' when 1 then 'New Business' when 2 then 'Permanent MTA'
             when 3 then 'Temp Add Driver' when 4 then 'Cancel Temp Add Driver'
             when 5 then 'Temporary Additional Vehicle' when 6 then 'Cancel Temporary Additional Vehicle'
             when 7 then 'Temp Substatution' when 8 then 'Cancel Temp Substatution' when 9 then 'Suspension'
             when 10 then 'Cancel Suspension' when 11 then 'Cancellation' when 12 then 'Reinstate'
             when 13 then 'Duplicate Certificate' when 14 then 'Renewal Offer' when 15 then 'Renewal Confirmation'
             when 16 then 'Lapse' when 17 then 'Insurer Led Cancelation' when 20 then 'Renewal Transfer'
             when 23 then 'Renewal Chase' when 24 then 'Document Chase' when 25 then 'Prior Year Quotes'
             when 26 then 'Accounts Business Process' when 27 then 'Amend NCD'
             else 'Unknown Business Process' end                    as BusinessProcess,
        a.EventType                                                 as EventTypeCode,
        d.DisplayName                                               as EventType,
        b.Description                                               as DocumentDescription,
        a.CreatedDate                                               as EventDate,
        c.Event_CreatedDate                                         as DocumentEventDate
from    ${ods.dlk}.relay_eventmodel_event           a
join    ${ods.dlk}.relay_eventmodel_documentevent   c   on c.EventId = a.EventId
join    ${ods.dlk}.relay_documentation_document     b   on b.DocumentId = c.DocumentId
left join (
        select  Code, DisplayName
        from    ${ods.dlk}.relay_reporting_relaycodelist
        where   DeletedDate is null and Category = 'Relay.EventModel.EventType'
        ) d     on d.Code = a.EventType
where   a.CreatedDate >= date '${ods.cutoff_date}'

In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.PolicyDocumentEvent                                                          */
/*  Description : Relay document events (offers, chases, confirmations, cancellations ...) on policies, */
/*              :  as notebook 05: event ->                                                             */
/*              : document -> policy document -> policy event -> policy -> portfolio. Business process  */
/*              : and event type are kept as                                                            */
/*              : code and description rather than one concatenated string. Business process names are  */
/*              : kept exactly as Chill's                                                               */
/*              : code has them, spelling included ('Temp Substatution', 'Insurer Led Cancelation'): co */
/*              : nsumers match on them.                                                                */
/*              :                                                                                       */
/*              : Grain: one row per Relay document event and policy                                    */
/*              : Business key: EventId + DocumentId + PolicyCode                                       */
/*              : History: event   Cut-off: EventDate                                                   */
/*              : Source: dlk.relay_eventmodel_* + dlk.relay_documentation_* + dlk.relay_policy / portf */
/*              : olio                                                                                  */
/*              : Reads: dlk.relay_documentation_policydocument, stg.odsv2_relay_document_event, dlk.re */
/*              : lay_eventmodel_policyevent, dlk.relay_policy, dlk.relay_portfolio, ods_v2.Policy      */
/******************************************************************************************************/
create or replace table ${ods.target}.PolicyDocumentEvent as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  distinct
                b.EventId,
                b.DocumentId,
                concat(upper(trim(e.Code)), lpad(cast(d.PolicySequence as string), 3, '0'))
                                                                                as PolicyCode,
                p.PolicyId,
                nullif(upper(trim(e.Code)), '')                                                    as ClientCode,
                d.PolicyId                                                      as RelayPolicyId,
                d.PolicySequence,
                b.BusinessProcessCode,
                b.BusinessProcess,
                b.EventTypeCode,
                b.EventType,
                b.DocumentDescription,
                b.EventDate,
                b.DocumentEventDate
        from    ${ods.dlk}.relay_documentation_policydocument   a
        join    ${ods.stg}.odsv2_relay_document_event           b   on b.DocumentId = a.DocumentId
        join    ${ods.dlk}.relay_eventmodel_policyevent         c   on c.EventId = upper(a.EventId)
        join    ${ods.dlk}.relay_policy                         d   on d.PolicyId = c.PolicyId
        join    ${ods.dlk}.relay_portfolio                      e   on e.PortfolioId = d.PortfolioId
        left join ${ods.target}.Policy                          p
                on  p.PolicyCode = concat(upper(trim(e.Code)), lpad(cast(d.PolicySequence as string), 3, '0'))
        ) x

## RenewalPortalEvent


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.RenewalPortalEvent                                                           */
/*  Description : Applied Renewals portal events: logins (successful / failed) and renewal payments (su */
/*              : ccessful / failed).                                                                   */
/*              :                                                                                       */
/*              : Grain: one row per portal event type and event id                                     */
/*              : Business key: EventType + EventId                                                     */
/*              : History: event   Cut-off: EventDateTime                                               */
/*              : Source: dlk.appliedrenewals_paymentsuccess / paymentfailedevent / successfullogineven */
/*              : t / failedloginevent                                                                  */
/*              : Reads: dlk.appliedrenewals_paymentsuccess, dlk.appliedrenewals_paymentfailedevent, dl */
/*              : k.appliedrenewals_successfulloginevent, dlk.appliedrenewals_failedloginevent          */
/******************************************************************************************************/
create or replace table ${ods.target}.RenewalPortalEvent as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select 'PaymentSuccess' as EventType, Id as EventId, SessionId, `Timestamp` as EventDateTime, nullif(upper(trim(PortfolioCode)), '') as PortfolioCode, nullif(upper(trim(PolicyCodeForRenewal)), '') as PolicyCode, PaymentAmount, PaymentType, cast(null as string) as PaymentErrorCode, cast(null as string) as PaymentErrorMessage, cast(null as boolean) as IsLocked from ${ods.dlk}.appliedrenewals_paymentsuccess where `Timestamp` >= date '${ods.cutoff_date}'
        union all
        select 'PaymentFailed', Id, SessionId, `Timestamp`, nullif(upper(trim(PortfolioCode)), ''), nullif(upper(trim(PolicyCodeForRenewal)), ''), PaymentAmount, PaymentType, PaymentErrorCode, PaymentErrorMessage, cast(null as boolean) from ${ods.dlk}.appliedrenewals_paymentfailedevent where `Timestamp` >= date '${ods.cutoff_date}'
        union all
        select 'LoginSuccess', Id, SessionId, `Timestamp`, nullif(upper(trim(PortfolioCode)), ''), cast(null as string), cast(null as decimal(18,2)), cast(null as string), cast(null as string), cast(null as string), cast(null as boolean) from ${ods.dlk}.appliedrenewals_successfulloginevent where `Timestamp` >= date '${ods.cutoff_date}'
        union all
        select 'LoginFailed', Id, SessionId, `Timestamp`, nullif(upper(trim(PortfolioCode)), ''), cast(null as string), cast(null as decimal(18,2)), cast(null as string), cast(null as string), cast(null as string), isLocked from ${ods.dlk}.appliedrenewals_failedloginevent where `Timestamp` >= date '${ods.cutoff_date}'
        ) x

## DocumentWorkflowStatus


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.DocumentWorkflowStatus                                                       */
/*  Description : MyChill document workflow (document requested / accepted / rejected per policy). Agen */
/*              : t and assigned-to emails are                                                          */
/*              : not taken; the customer's email is a salted hash.                                     */
/*              :                                                                                       */
/*              : Grain: one row per MyChill workflow record (pk + rk)                                  */
/*              : Business key: Pk + Rk                                                                 */
/*              : History: event (no cut-off; small)   Cut-off: none                                    */
/*              : Source: dlk.mychillworkflow_documentstatus                                            */
/*              : Reads: dlk.mychillworkflow_documentstatus                                             */
/******************************************************************************************************/
create or replace table ${ods.target}.DocumentWorkflowStatus as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  pk                                                              as Pk,
                rk                                                              as Rk,
                GUID                                                            as WorkflowGuid,
                nullif(upper(trim(PolicyCode)), '')                                                as PolicyCode,
                DiaryTypeID,
                Title,
                Attempts,
                IsAccepted,
                IsRejected,
                ods_redact(Reasons)                                                 as Reasons,
                AssignedDate,
                `Timestamp`                                                     as EventTimestamp,
                InsertDateTime,
                ods_hash_email(CustomerEmail)                                       as CustomerEmailKey
        from    ${ods.dlk}.mychillworkflow_documentstatus
        ) x

## DocumentUpload


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.DocumentUpload                                                               */
/*  Description : MyChill document uploads per policy. DocumentValue (the value keyed on the document,  */
/*              : e.g. a licence or PPS                                                                 */
/*              : number) is not taken.                                                                 */
/*              :                                                                                       */
/*              : Grain: one row per MyChill upload record (pk + rk)                                    */
/*              : Business key: Pk + Rk                                                                 */
/*              : History: event (no cut-off; small)   Cut-off: none                                    */
/*              : Source: dlk.mychill_newuploaddocumentevents                                           */
/*              : Reads: dlk.mychill_newuploaddocumentevents                                            */
/******************************************************************************************************/
create or replace table ${ods.target}.DocumentUpload as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  pk                                                              as Pk,
                rk                                                              as Rk,
                Id                                                              as UploadId,
                nullif(upper(trim(PolicyCode)), '')                                                as PolicyCode,
                Title,
                DiaryCode,
                Type                                                            as DocumentType,
                PagesUploaded,
                IsUploaded,
                ErrorMEssage                                                    as ErrorMessage,
                `Timestamp`                                                     as EventTimestamp,
                InsertDateTime
        from    ${ods.dlk}.mychill_newuploaddocumentevents
        ) x

## DocumentChaseHome


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.DocumentChaseHome                                                            */
/*  Description : Daily MyChill document chase snapshot for home policies: each required document's dat */
/*              : e and status, and the chase                                                           */
/*              : send dates. Names and mobile number are not taken.                                    */
/*              :                                                                                       */
/*              : Grain: one row per file date and policy code                                          */
/*              : Business key: FileDate + PolicyCode                                                   */
/*              : History: snapshot (daily)   Cut-off: CreateDate                                       */
/*              : Source: edw.exp_mychill_chase_daily_snapshot_home                                     */
/*              : Reads: edw.exp_mychill_chase_daily_snapshot_home                                      */
/******************************************************************************************************/
create or replace table ${ods.target}.DocumentChaseHome as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  FileDate, nullif(upper(trim(PolicyCode)), '') as PolicyCode, PolicyStatus, PolicyTypeGroup, PolicyType, SaleDate, InceptionDate,
                Comp_DDM_Date, Comp_DDM_Status, Gap_In_Cov_Ltr_Date, Gap_In_Cov_Ltr_Status,
                Val_For_Spec_Item_Date, Val_For_Spec_Item_Status, PPS_Num_Date, PPS_Num_Status,
                Identification_Date, Identification_Status, Digital_Journey_Date, Digital_Journey_Status,
                Finance_Form_Date, Finance_Form_Status, Campaign, Day1SendDate, Day7SendDate, Day14SendDate, Day20SendDate,
                CreateDate
        from    ${ods.edw}.exp_mychill_chase_daily_snapshot_home
        where   CreateDate >= date '${ods.cutoff_date}'
        ) x

## DocumentChaseMotorVan


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.DocumentChaseMotorVan                                                        */
/*  Description : Daily MyChill document chase snapshot for motor and van policies (the source repeats  */
/*              : a few file-date + policy                                                              */
/*              : rows: latest kept). Names and mobile number are not taken.                            */
/*              :                                                                                       */
/*              : Grain: one row per file date and policy code                                          */
/*              : Business key: FileDate + PolicyCode                                                   */
/*              : History: snapshot (daily)   Cut-off: CreateDate                                       */
/*              : Source: edw.exp_mychill_chase_daily_snapshot_motorvan                                 */
/*              : Reads: edw.exp_mychill_chase_daily_snapshot_motorvan                                  */
/******************************************************************************************************/
create or replace table ${ods.target}.DocumentChaseMotorVan as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  FileDate, PolicyCodeClean as PolicyCode, PolicyStatus, PolicyTypeGroup, PolicyType, SaleDate, InceptionDate,
                Sign_Prop_Date, Sign_Prop_Status, Sign_Stat_Of_Fact_Date, Sign_Stat_Of_Fact_Status,
                Comp_Fin_Agree_Date, Comp_Fin_Agree_Status, Comp_DDM_Date, Comp_DDM_Status,
                Proof_Of_NCB_Date, Proof_Of_NCB_Status, Trans_Proof_Of_NCB_Date, Trans_Proof_Of_NCB_Status,
                Ltr_Of_Driv_Exp_Date, Ltr_Of_Driv_Exp_Status, Prop_Driv_Lic_Date, Prop_Driv_Lic_Status,
                Nam_Driv_Lic_Date, Nam_Driv_Lic_Status, Gap_In_Cov_Ltr_Date, Gap_In_Cov_Ltr_Status,
                `2ndCar_Cert_Date` as SecondCar_Cert_Date, `2ndCar_Cert_Status` as SecondCar_Cert_Status,
                Doc_Ltr_Date, Doc_Ltr_Status, Engineers_Rpt_Date, Engineers_Rpt_Status, NCT_Date, NCT_Status,
                Veh_Lic_Cert_Date, Veh_Lic_Cert_Status, Irish_Reg_Date, Irish_Reg_Status,
                Main_Driver_Dec_Date, Main_Driver_Dec_Status, Soc_Dom_Pleas_Dec_Date, Soc_Dom_Pleas_Dec_Status,
                Comp_Car_Exper_Date, Comp_Car_Exper_Status, DD_Conf_Ltr_Date, DD_Conf_Ltr_Status,
                Comp_Fin_Mand_Date, Comp_Fin_Mand_Status, Val_For_Spec_Item_Date, Val_For_Spec_Item_Status,
                Dri_Lic_Num_Date, Dri_Lic_Num_Status, Comp_Lost_Cert_Dec_Date, Comp_Lost_Cert_Dec_Status,
                Orig_Cert_Date, Orig_Cert_Status, Cancel_Req_Date, Cancel_Req_Status, PPS_Num_Date, PPS_Num_Status,
                Afford_State_Date, Afford_State_Status, Identification_Date, Identification_Status, InsurerId,
                Digital_Journey_Date, Digital_Journey_Status, Finance_Form_Date, Finance_Form_Status,
                Nam_Driv_Lic_Date2, Nam_Driv_Lic_Status2, Nam_Driv_Lic_Date3, Nam_Driv_Lic_Status3,
                Nam_Driv_Lic_Date4, Nam_Driv_Lic_Status4, Nam_Driv_Lic_Date5, Nam_Driv_Lic_Status5,
                Campaign, Day1SendDate, Day7SendDate, Day14SendDate, Day20SendDate, CreateDate
        from    (
                select  *, nullif(upper(trim(PolicyCode)), '') as PolicyCodeClean,
                        row_number() over (partition by FileDate, upper(trim(PolicyCode)) order by CreateDate desc) as rn
                from    ${ods.edw}.exp_mychill_chase_daily_snapshot_motorvan
                where   CreateDate >= date '${ods.cutoff_date}'
                ) m
        where   rn = 1
        ) x

## Checks


In [ ]:
%%pyspark
# Checks for the tables this notebook built. All are checked and logged to OdsBuildLog; the cell then fails if
# any check failed, naming the table and the likely cause.
import json, re
from functools import reduce
from pyspark.sql import functions as F
SPEC = json.loads(r"""[{"table": "Policy", "bk": ["PolicyCode"], "sk": "PolicyId", "nullable": [], "allow": []}, {"table": "PolicyVersion", "bk": ["PolicyCode", "RecordVersion"], "sk": "PolicyVersionId", "nullable": [], "allow": []}, {"table": "PolicyTravel", "bk": ["MapfrePolicyId"], "sk": null, "nullable": [], "allow": []}, {"table": "PolicyTravelTransaction", "bk": ["TravelTransactionKey"], "sk": null, "nullable": [], "allow": []}, {"table": "PolicyMotorProposer", "bk": ["PolicyCode", "RecordVersion"], "sk": null, "nullable": [], "allow": []}, {"table": "PolicyMotorVehicle", "bk": ["PolicyCode", "RecordVersion"], "sk": null, "nullable": [], "allow": []}, {"table": "PolicyHome", "bk": ["PolicyCode", "RecordVersion"], "sk": null, "nullable": [], "allow": []}, {"table": "PolicySale", "bk": ["PolicyCode", "ReportingSaleDate"], "sk": null, "nullable": [], "allow": []}, {"table": "PolicyRenewal", "bk": ["PolicyCode", "LYPolicyRenewDateAdj"], "sk": null, "nullable": [], "allow": []}, {"table": "PolicyMonthEnd", "bk": ["PolicyCode", "EffectiveDate"], "sk": null, "nullable": [], "allow": []}, {"table": "PolicyCancellation", "bk": ["PolicyCancellationId"], "sk": null, "nullable": [], "allow": []}, {"table": "PolicyTransaction", "bk": ["PolicyCode", "PostingDate", "TransSeq", "ArCode", "ArNumber", "ArReference", "ClientTransType", "ClientAmount"], "sk": "PolicyTransactionId", "nullable": [], "allow": []}, {"table": "PolicyMta", "bk": ["PolicyCode", "HistorySeq", "HistoryType"], "sk": null, "nullable": [], "allow": []}, {"table": "PolicyClaimMotor", "bk": ["PolicyCode", "ClaimDateRaw", "ClaimType", "ClaimValue", "ClaimSlot"], "sk": null, "nullable": [], "allow": []}, {"table": "PolicyClaimHome", "bk": ["HomeClaimId"], "sk": null, "nullable": [], "allow": []}, {"table": "ClientPortfolio", "bk": ["PortfolioId", "PolicyCode"], "sk": null, "nullable": [], "allow": []}, {"table": "PolicyDocumentEvent", "bk": ["EventId", "DocumentId", "PolicyCode"], "sk": null, "nullable": [], "allow": []}, {"table": "RenewalPortalEvent", "bk": ["EventType", "EventId"], "sk": null, "nullable": [], "allow": []}, {"table": "DocumentWorkflowStatus", "bk": ["Pk", "Rk"], "sk": null, "nullable": [], "allow": []}, {"table": "DocumentUpload", "bk": ["Pk", "Rk"], "sk": null, "nullable": [], "allow": []}, {"table": "DocumentChaseHome", "bk": ["FileDate", "PolicyCode"], "sk": null, "nullable": [], "allow": []}, {"table": "DocumentChaseMotorVan", "bk": ["FileDate", "PolicyCode"], "sk": null, "nullable": [], "allow": []}]""")
PII_NAME = re.compile(r"(e_?mail|forename|first_?name|surname|last_?name|full_?name|second_?name|phone|mobile|telephone|post_?code|eircode(?!_?area)|address_?line|risk_?address|postal_?address|dob$|date_?of_?birth|licen[cs]e_?num|registration_?no|reg_?number|registration_number|^ani$|^dnis$|originatingdnis|^payer_|iban|^bic|^contact$|rendered_?message)", re.I)
EMAIL_RE = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
MOBILE_RE = r"(^|[^0-9A-Za-z-])(\+?353 ?|0)8[35-9][ -]?[0-9]{3}[ -]?[0-9]{4}([^0-9A-Za-z-]|$)"   # stands alone, not inside an id
rows, failures = [], []
for s in SPEC:
    t = f"{target_schema}.{s['table']}"
    df = spark.table(t)
    bk = s["bk"]
    must = [c for c in bk if c not in s["nullable"]]
    agg = df.agg(F.count(F.lit(1)).alias("n"),
                 F.countDistinct(F.struct(*bk)).alias("d_bk"),
                 F.sum(F.when(reduce(lambda a, b: a | b, [F.col(c).isNull() for c in must] or [F.lit(False)]), 1).otherwise(0)).alias("null_bk"),
                 *( [F.countDistinct(s["sk"]).alias("d_sk")] if s["sk"] else [] )).first()
    pii_cols = [c for c in df.columns if PII_NAME.search(c) and not c.endswith("Key") and c not in s["allow"]]
    text_cols = [f.name for f in df.schema.fields if f.dataType.simpleString() == "string"]   # hashes never match
    hits = {}
    if text_cols:
        found = df.select(*text_cols).limit(1000000).agg(
            *[F.sum((F.col(c).rlike(EMAIL_RE) | F.col(c).rlike(MOBILE_RE)).cast("int")).alias(c) for c in text_cols]).first().asDict()
        hits = {c: v for c, v in found.items() if v}
    problems, advice = [], []
    keys = " + ".join(bk)
    if agg["n"] == 0:
        problems.append("empty")
        advice.append("the table is empty: its source is empty, or a filter (cut-off date, join) removed every row")
    if agg["d_bk"] != agg["n"]:
        problems.append(f"{agg['n'] - agg['d_bk']:,} duplicate business keys")
        advice.append(f"several rows share a key ({keys}): the source sent the same record more than once, or the "
                      f"dedupe rule in the cell no longer holds. Examples: select {', '.join(bk)}, count(*) from {t} "
                      f"group by {', '.join(bk)} having count(*) > 1")
    if agg["null_bk"]:
        problems.append(f"{agg['null_bk']:,} null business keys")
        advice.append(f"key columns ({', '.join(must)}) are empty on some rows: if that is legitimate for a column, add it "
                      f"to key_nullable for this table in the config; otherwise the source changed")
    if s["sk"] and agg["d_sk"] != agg["n"]:
        problems.append("surrogate key not unique")
        advice.append("the surrogate key repeats: the business key is not unique (see above)")
    if pii_cols:
        problems.append(f"PII-named columns {pii_cols}")
        advice.append("columns named like personal data: hash them (HASH_*) or leave them out in the config")
    if hits:
        problems.append(f"email/mobile values in {hits}")
        advice.append("emails / mobile numbers inside these columns: wrap the column in REDACT() (or hash it) in the config")
    rows.append((batch_id, s["table"], agg["n"], agg["n"] - agg["d_bk"], agg["null_bk"], "; ".join(problems) or "ok"))
    print(f"{s['table']:<28} {agg['n']:>13,}  {'; '.join(problems) or 'ok'}")
    if problems:
        failures.append(f"{s['table']}: " + " | ".join(advice))
log = spark.createDataFrame(rows, "OdsBatchId string, TableName string, RowCount long, DuplicateKeys long, NullKeys long, Result string")
log.withColumn("CheckedAt", F.current_timestamp()).write.mode("append").saveAsTable(f"{target_schema}.OdsBuildLog")
if failures:
    ods_run_log("10_Policy", "failed", "; ".join(f.split(":")[0] for f in failures))
    raise AssertionError("ods checks failed:\n- " + "\n- ".join(failures))
ods_run_log("10_Policy", "succeeded")
